## Camada Bronze: Ingestão de Dados Brutos

In [8]:
import pandas as pd
import requests
from google.cloud import bigquery

# Configurações do ambiente GCP
PROJECT_ID = 'gcp-pratica' # @param {type:"string"}
DATASET_ID = 'dataset_pratica'
USER_ID = 'user_data_eng'
TABLE_NAME_BRONZE = f'tb_bronze_facts_{USER_ID}'
TABLE_ID_BRONZE = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME_BRONZE}'

# 1. Fetch data from Useless Facts API (coletando 20 fatos)
print('Fetching data from Useless Facts API...')
url = 'https://uselessfacts.jsph.pl/api/v2/facts/random?language=en'

facts = []
for _ in range(20):
    res = requests.get(url)
    if res.status_code == 200:
        facts.append(res.json())

# 2. Process into DataFrame
df_bronze = pd.DataFrame(facts)[['id', 'text', 'source']]
print(f'Ingested {len(df_bronze)} rows.')
print(df_bronze.head())

Fetching data from Useless Facts API...
Ingested 20 rows.
                                 id  \
0  f2e92a6588058c8d14487fb0c26a3f34   
1  bab733d57a50b575600048330c59867d   
2  51ca37a7f7fde2bcb0392c6d6d63fe45   
3  6a0c05a68e09a99b9f6860ac1e92a7fe   
4  a542fde858bdc93ce40674232ec86a1f   

                                                text      source  
0  Celery has negative calories! It takes more ca...  djtech.net  
1  On average, 12 newborns will be given to the w...  djtech.net  
2  The 57 on the Heinz ketchup bottle represents ...  djtech.net  
3  If one places a tiny amount of liquor on a sco...  djtech.net  
4  The average American works 24,000 hours in the...  djtech.net  


In [9]:
import time

# 1. (Opcional) Limpa a tabela antiga para evitar qualquer conflito de schema
client.delete_table(TABLE_ID_BRONZE, not_found_ok=True)

# 2. Configura o job
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# 3. Executa o carregamento com tratamento de tentativas (Retry Loop)
max_tentativas = 3
tempo_espera = 3  # segundos

for tentativa in range(1, max_tentativas + 1):
    try:
        job = client.load_table_from_dataframe(
            df_bronze,
            TABLE_ID_BRONZE,
            job_config=job_config
        )
        job.result()  # Aguarda a conclusão do Job
        print(f'✅ Sucesso! {len(df_bronze)} linhas carregadas em {TABLE_ID_BRONZE}')
        break  # Sai do loop se der certo
    except Exception as e:
        print(f'⚠️ Tentativa {tentativa}/{max_tentativas} falhou devido a instabilidade ({e}).')
        if tentativa == max_tentativas:
            print('❌ Limite de tentativas atingido.')
            raise e
        print(f'Aguardando {tempo_espera} segundos para tentar novamente...')
        time.sleep(tempo_espera)

✅ Sucesso! 20 linhas carregadas em gcp-pratica.dataset_pratica.tb_bronze_facts_user_data_eng


## Camada Silver: Enriquecimento e Limpeza

In [10]:
import vertexai
from vertexai.generative_models import GenerativeModel

# Configurações da Camada Silver
TABLE_NAME_SILVER = f'tb_silver_facts_enriched_{USER_ID}'
TABLE_ID_SILVER = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME_SILVER}'

# Inicializar Vertex AI
vertexai.init(project=PROJECT_ID, location='us-central1')
model = GenerativeModel('gemini-2.5-flash')

# 1. Carregar dados da Bronze
query_bronze = f'SELECT * FROM `{TABLE_ID_BRONZE}`'
df_silver = client.query(query_bronze).to_dataframe()

# 2. Função de enriquecimento com LLM (Classificação por tema)
def classificar_fato(texto):
    prompt = f'Analise o fato curioso a seguir e classifique-o em apenas UMA das categorias: Ciência, História, Geografia, Biologia, Pop Culture ou Geral. Responda APENAS a palavra da categoria: "{texto}"'
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except:
        return 'Não Classificado'

print('🤖 Enriquecendo fatos curiosos com Gemini...')
df_silver['categoria'] = df_silver['text'].apply(classificar_fato)

# 3. Salvar na tabela Silver
client.load_table_from_dataframe(df_silver, TABLE_ID_SILVER, job_config=job_config).result()
print(f'✅ Camada Silver concluída: {TABLE_ID_SILVER}')
display(df_silver.head())

/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


🤖 Enriquecendo fatos curiosos com Gemini...
✅ Camada Silver concluída: gcp-pratica.dataset_pratica.tb_silver_facts_enriched_user_data_eng


,id,text,source,categoria
0,f2e92a6588058c8d14487fb0c26a3f34,Celery has negative calories! It takes more ca...,djtech.net,Biologia
1,bab733d57a50b575600048330c59867d,"On average, 12 newborns will be given to the w...",djtech.net,Geral
2,51ca37a7f7fde2bcb0392c6d6d63fe45,The 57 on the Heinz ketchup bottle represents ...,djtech.net,História
3,6a0c05a68e09a99b9f6860ac1e92a7fe,If one places a tiny amount of liquor on a sco...,djtech.net,Biologia
4,a542fde858bdc93ce40674232ec86a1f,"The average American works 24,000 hours in the...",djtech.net,Geral


## Camada Gold: Preparação para Negócio e Visualização

In [11]:
import plotly.graph_objects as go

# 1. Preparação dos dados para Pareto
df_gold = df_silver['categoria'].value_counts().reset_index()
df_gold.columns = ['categoria', 'quantidade']
df_gold = df_gold.sort_values(by='quantidade', ascending=False)

df_gold['percentual_acumulado'] = (df_gold['quantidade'].cumsum() / df_gold['quantidade'].sum()) * 100

# 2. Visualização
fig = go.Figure()
fig.add_trace(go.Bar(x=df_gold['categoria'], y=df_gold['quantidade'], name='Frequência'))
fig.add_trace(go.Scatter(x=df_gold['categoria'], y=df_gold['percentual_acumulado'], name='% Acumulada', yaxis='y2'))

fig.update_layout(
    title='Análise de Pareto - Feedback de Clientes (Camada Gold)',
    yaxis=dict(title='Quantidade'),
    yaxis2=dict(title='Percentual (%)', overlaying='y', side='right', range=[0, 110]),
    template='plotly_white'
)

fig.show()

In [12]:
import plotly.express as px

# 1. Enriquecimento de métrica de texto na Gold
df_silver['qtd_palavras'] = df_silver['text'].apply(lambda x: len(str(x).split()))

df_gold_complexidade = (
    df_silver.groupby('categoria')['qtd_palavras']
    .mean()
    .reset_index()
    .sort_values(by='qtd_palavras', ascending=False)
)

# 2. Visualização
fig_complexidade = px.bar(
    df_gold_complexidade,
    x='categoria',
    y='qtd_palavras',
    title='Complexidade de Conteúdo: Média de Palavras por Categoria',
    labels={'categoria': 'Categoria', 'qtd_palavras': 'Média de Palavras'},
    color='qtd_palavras',
    color_continuous_scale='Teal'
)

fig_complexidade.update_layout(template='plotly_white')
fig_complexidade.show()